# GalactICS → ntropy: NFW halo end-to-end

*A walkthrough notebook for the PyGalactICS rewrite — from graduate-school Fortran to a testable Python library.*

This notebook demonstrates the **integrated workflow** the rewrite was built for:

1. **galacticsics** — define an NFW halo model, run legacy `dbh` (self-consistent potential + DF tables), sample particles with `genhalo`.
2. **ntropy** — evolve those particles with self-gravity; validate force methods and density stability.

Steps covered:

- Build a GalactICS NFW halo model and sample ICs through the real `dbh` + `genhalo` pipeline
- Compare **brute-force** vs **Barnes–Hut** force accuracy at several opening angles θ
- Plot **ρ(r)** before/after a short N-body run

Artifacts → `notebooks/artifacts/` (gitignored).

**Prerequisites:** `make install-dev` (builds `legacy/bin`, installs both packages), plus `pip install jupyter matplotlib`.

## Units and layout

The Python packages in this repo share **GalactICS internal units** (`G=1`):

| Quantity | Unit |
|----------|------|
| Length | kpc |
| Velocity | 100 km/s |
| Mass | 2.325×10⁹ M☉ |

- **`galacticsics`** — Python API + subprocess bridge to legacy `dbh`, `genhalo`, `gendisk`, …
- **`ntropy`** — minimal N-body code that *consumes* GalactICS particle files

```
GalaxyModel  →  dbh (solve)  →  genhalo  →  ParticleSet  →  ntropy Simulation
```

Fortran/C sources live under `legacy/` (gitignored, compiled by `make legacy-build`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

from ntropy.analysis.density import bin_spherical_density, compare_density_profiles
from ntropy.config import ForceConfig, IntegratorConfig, ParallelConfig, RunConfig
from ntropy.forces.brute import compute_forces_brute
from ntropy.forces.bhtree import compute_forces_bh
from ntropy.integrations.galacticsics import (
    galacticsics_available,
    nfw_halo_model_fast,
    sample_galacticsics_halo,
)
from ntropy.simulation import Simulation

if not galacticsics_available():
    raise RuntimeError(
        "galacticsics + legacy binaries required. From repo root: make install-dev"
    )

# Repo root (parent of notebooks/)
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
elif not (REPO / "src" / "ntropy").exists():
    REPO = REPO.parent  # running from notebooks/ subdir in some IDEs

ARTIFACTS = REPO / "notebooks" / "artifacts" / "nfw_walkthrough"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Keep matplotlib cache inside artifacts (useful in CI / sandboxed envs)
os.environ.setdefault("MPLCONFIGDIR", str(ARTIFACTS / ".matplotlib"))

import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print(f"Artifacts → {ARTIFACTS}")

## 1. GalactICS: model → dbh → genhalo

We use the same path as production GalactICS:

1. **`GalaxyModel`** — spherical NFW halo (`lmax=0`, coarse grid for speed)
2. **`solve_potential`** — runs `legacy/bin/dbh`, writes `dbh.dat`, `dfnfw.dat`, …
3. **`genhalo`** — samples equilibrium halo particles from the Eddington DF tables

`ntropy.integrations.galacticsics.sample_galacticsics_halo()` wraps these three steps and returns a `ParticleState`.

In [ ]:
model = nfw_halo_model_fast()
print(model)

GALACTICS_WORK = ARTIFACTS / "galacticsics_work"
EPS = 0.04

ic_result = sample_galacticsics_halo(
    model,
    n_particles=256,
    seed=-42,  # legacy ran3 convention (negative integer)
    work_dir=GALACTICS_WORK,
    eps=EPS,
    solve=True,
)

state = ic_result.state
particle_file = ARTIFACTS / "nfw_halo_galacticsics.dat"
state.write_ascii(particle_file)

print(f"N = {state.n}, total mass = {state.mass.sum():.2f}")
print(f"GalactICS work_dir = {ic_result.work_dir}")
print(f"dbh.dat present: {(ic_result.work_dir / 'dbh.dat').exists()}")
print(f"dfnfw.dat present: {(ic_result.work_dir / 'dfnfw.dat').exists()}")
print(f"Wrote {particle_file.name}")

## 2. Density profile from GalactICS particles

The halo DF comes from **`dbh`** (Eddington inversion of the NFW density embedded in the multipole solver). We bin the **genhalo** particles into spherical shells to estimate ρ(r).

In [ ]:
r_max = float(np.sqrt(np.sum(state.pos**2, axis=1)).max()) * 1.1
profile = bin_spherical_density(state.pos, state.mass, n_bins=20, r_max=r_max)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(profile.r_mid, profile.rho, "o-", ms=5, alpha=0.8, label="genhalo particles")
ax.set_xlabel("r [kpc]")
ax.set_ylabel("ρ [M_unit / kpc³]")
ax.set_title("GalactICS NFW halo — initial ρ(r)")
ax.legend()
fig.tight_layout()
density_init_path = ARTIFACTS / "density_initial.png"
fig.savefig(density_init_path, dpi=150)
plt.show()
print(f"Saved {density_init_path.name}")

## 3. Force accuracy: brute force vs Barnes–Hut

**Brute force** is our gold standard (O(N²)). The **Barnes–Hut tree** approximates distant groups with monopole nodes; the opening angle **θ** controls accuracy vs speed (`size/distance < θ` → open the node).

Below we measure the maximum relative acceleration error ‖a_BH − a_brute‖ / ‖a_brute‖ for several θ values.

In [ ]:
pos, mass, eps = state.pos, state.mass, state.eps
acc_brute = compute_forces_brute(pos, mass, eps)
brute_norm = np.linalg.norm(acc_brute, axis=1)
brute_norm = np.maximum(brute_norm, 1e-30)

thetas = [0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
rows = []

for theta in thetas:
    acc_bh = compute_forces_bh(pos, mass, eps, theta=theta)
    delta = acc_bh - acc_brute
    rel = np.linalg.norm(delta, axis=1) / brute_norm
    rows.append({
        "theta": theta,
        "max_rel": float(rel.max()),
        "median_rel": float(np.median(rel)),
        "rms_rel": float(np.sqrt(np.mean(rel**2))),
    })

print(f"{'theta':>6} {'max_rel':>10} {'median_rel':>12} {'rms_rel':>10}")
for row in rows:
    print(f"{row['theta']:6.1f} {row['max_rel']:10.4f} {row['median_rel']:12.4f} {row['rms_rel']:10.4f}")

csv_path = ARTIFACTS / "force_accuracy.csv"
with open(csv_path, "w") as f:
    f.write("theta,max_rel,median_rel,rms_rel\n")
    for row in rows:
        f.write(f"{row['theta']},{row['max_rel']},{row['median_rel']},{row['rms_rel']}\n")
print(f"\nSaved {csv_path.name}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

theta_arr = [r["theta"] for r in rows]
axes[0].semilogy(theta_arr, [r["max_rel"] for r in rows], "o-", label="max")
axes[0].semilogy(theta_arr, [r["median_rel"] for r in rows], "s-", label="median")
axes[0].set_xlabel("opening angle θ")
axes[0].set_ylabel("relative |Δa| / |a_brute|")
axes[0].set_title("BH force error vs θ")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-particle error at θ = 0.5
theta_demo = 0.5
acc_demo = compute_forces_bh(pos, mass, eps, theta=theta_demo)
rel_demo = np.linalg.norm(acc_demo - acc_brute, axis=1) / brute_norm
r = np.sqrt(np.sum(pos**2, axis=1))
axes[1].scatter(r, rel_demo, s=12, alpha=0.6, c=r, cmap="viridis")
axes[1].set_xlabel("r [kpc]")
axes[1].set_ylabel("relative force error")
axes[1].set_title(f"Per-particle error at θ = {theta_demo}")
fig.colorbar(axes[1].collections[0], ax=axes[1], label="r [kpc]")
fig.tight_layout()
force_plot_path = ARTIFACTS / "force_accuracy.png"
fig.savefig(force_plot_path, dpi=150)
plt.show()
print(f"Saved {force_plot_path.name}")

## 4. Short N-body run: does ρ(r) stay put?

This is what **ntropy** is for: a quick self-gravity check that your IC is not wildly out of equilibrium. We use brute force (exact for this N) and a modest timestep.

In [ ]:
cfg = RunConfig()
cfg.integrator = IntegratorConfig(dt=0.002, n_steps=25)
cfg.force = ForceConfig(method="brute")
cfg.parallel = ParallelConfig(enabled=False)
cfg.output.write_final = False
cfg.output.every = 0

result = Simulation(cfg, state=state.copy()).run()

init_prof = bin_spherical_density(
    result.initial_state.pos, result.initial_state.mass, n_bins=20, r_max=r_max
)
final_prof = bin_spherical_density(
    result.final_state.pos, result.final_state.mass, n_bins=20, r_max=r_max
)
max_drift = compare_density_profiles(init_prof, final_prof, min_count=3)
print(f"Max relative spherical density change after {cfg.integrator.n_steps} steps: {max_drift:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(init_prof.r_mid, init_prof.rho, "o-", label="t = 0 (GalactICS IC)")
ax.loglog(final_prof.r_mid, final_prof.rho, "s-", label=f"t = {cfg.integrator.n_steps} dt (ntropy)")
ax.set_xlabel("r [kpc]")
ax.set_ylabel("ρ [M_unit / kpc³]")
ax.set_title("Density stability — GalactICS IC + ntropy evolution")
ax.legend()
fig.tight_layout()
density_final_path = ARTIFACTS / "density_stability.png"
fig.savefig(density_final_path, dpi=150)
plt.show()
print(f"Saved {density_final_path.name}")

## 5. Blog series — where to go next

This notebook is intentionally small and reproducible. Future posts in the series might cover:

| Topic | Package | Notes |
|-------|---------|-------|
| Disk + halo from reference artifacts | `galacticsics` → `ntropy` | `sample_galacticsics_galaxy()` + `tests/generated/reference` |
| Halo-first two-step workflow | `galacticsics` | `examples/halo_first_workflow.py` |
| Exponential disk Σ(R) tests | `galacticsics` gendisk → `ntropy` | `test_galacticsics_integration.py` |
| MPI domain decomposition | `ntropy` | `mpirun -n 4 ntropy-run config.json` |
| Fitting NFW to particles | `galacticsics.fitting` | observational rotation curves |

**Cursor angle:** the rewrite isolates Fortran under `legacy/`, exposes typed Python APIs, replaces checked-in golden files with generated artifacts, and uses **integrated tests** (`ntropy/tests/test_galacticsics_integration.py`) to prove the old IC code still works in the new stack.

> **Note:** `ntropy.ics.*` provides pure-Python analytic samplers for fast unit tests without a legacy build. Production validation should use the **galacticsics** path shown here.

All artifacts from this run live under `notebooks/artifacts/nfw_walkthrough/` and are safe to delete and regenerate.